In [ ]:
!pip install scipy scikit-image

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session


In [ ]:
engine = create_engine(
    "sqlite+pysqlite:///chinook.db",
    echo=False
)

print(engine)

In [ ]:
metadata = MetaData()

metadata.reflect(engine)

In [ ]:
for table_name, table in metadata.tables.items():

    print(table_name)
    print(table.columns.keys())

    print('-' * 30)

In [ ]:
from sqlalchemy.ext.automap import automap_base

Base = automap_base(metadata=metadata)

Base.prepare()

In [ ]:
print(Base.classes.keys())

In [ ]:
['Album', 'Artist', 'Track', 'Invoice', ...]

In [ ]:
Track = Base.classes.Track

In [ ]:
with Session(engine) as session:

    tracks = session.query(Track).limit(5).all()

    for t in tracks:
        print(t.Name)

In [ ]:
# ============================================================
# EXERCÍCIO PRÁTICO - APIs NASA
# ============================================================

# Instale as bibliotecas caso necessário:
# !pip install requests matplotlib scikit-image

# ============================================================
# 1) IMPORTS
# ============================================================

import requests
import matplotlib.pyplot as plt

from skimage import io


# ============================================================
# 2) LENDO A CHAVE DA API
# ============================================================

# Arquivo key.json:
# API_KEY=SUA_CHAVE

with open('key.json', 'r') as f:
    API_KEY = f.read().split('=')[1].strip()

print(API_KEY)


# ============================================================
# 3) APOD - ASTRONOMY PICTURE OF THE DAY
# ============================================================

url_apod = 'https://api.nasa.gov/planetary/apod'

params = {
    'api_key': API_KEY
}

response = requests.get(url_apod, params=params)

print('Status code:', response.status_code)

dados = response.json()


# ============================================================
# 4) IMPRIMINDO CAMPOS
# ============================================================

print('\nCOPYRIGHT:\n')
print(dados.get('copyright'))

print('\nEXPLANATION:\n')
print(dados.get('explanation'))


# ============================================================
# 5) MOSTRANDO A IMAGEM
# ============================================================

# Algumas respostas possuem hdurl
# outras apenas url

img_url = dados.get('hdurl', dados.get('url'))

img = io.imread(img_url)

plt.figure(figsize=(12, 8))

plt.imshow(img)

plt.axis('off')

plt.title(dados.get('title'))

plt.show()


# ============================================================
# 6) LIMITES DA API
# ============================================================

print('\nHEADERS:\n')
print(response.headers)

print('\nLIMITE TOTAL:')
print(response.headers.get('X-RateLimit-Limit'))

print('\nLIMITE RESTANTE:')
print(response.headers.get('X-RateLimit-Remaining'))


# ============================================================
# 7) MANIFESTO DO ROVER CURIOSITY
# ============================================================

rover = 'curiosity'

url_manifest = (
    f'https://api.nasa.gov/mars-photos/api/v1/manifests/{rover}'
)

params = {
    'api_key': API_KEY
}

response = requests.get(url_manifest, params=params)

manifesto = response.json()

photo_manifest = manifesto['photo_manifest']


# ============================================================
# 8) EXTRAINDO max_sol E max_date
# ============================================================

max_sol = photo_manifest['max_sol']
max_date = photo_manifest['max_date']

print('\nMAX SOL:')
print(max_sol)

print('\nMAX DATE:')
print(max_date)


# ============================================================
# 9) COLETANDO FOTOS DO ROVER
# ============================================================

url_photos = (
    f'https://api.nasa.gov/mars-photos/api/v1/rovers/{rover}/photos'
)

# câmeras desejadas
cameras_desejadas = ['NAVCAM', 'FHAZ', 'RHAZ']

page = 1

while True:

    params = {
        'api_key': API_KEY,
        'sol': max_sol,
        'page': page
    }

    response = requests.get(url_photos, params=params)

    dados = response.json()

    fotos = dados.get('photos')

    # encerra quando não houver fotos
    if not fotos:
        print('\nFim das páginas.')
        break

    print(f'\nPágina {page} - {len(fotos)} fotos')

    for foto in fotos:

        camera = foto['camera']['name']

        # filtra apenas as câmeras desejadas
        if camera in cameras_desejadas:

            img_url = foto['img_src']

            try:

                img = io.imread(img_url)

                plt.figure(figsize=(8, 8))

                plt.imshow(img)

                plt.axis('off')

                plt.title(
                    f'Página {page} | '
                    f'Câmera: {camera} | '
                    f'ID: {foto["id"]}'
                )

                plt.show()

            except Exception as e:
                print('Erro ao carregar imagem:', e)

    page += 1